In [1]:
from VAE.VQ_VAE import VQVAE
from Pixel_CNN.pixel_cnn import PixelCNN
from Pixel_CNN.pixel_cnn_utils import train, optimizer
import torch
from datasets.latent_dataset import LatentIndicesDataset
from image_generator import reconstruct_from_indices, save_image

import json

### Hyperparams

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

with open("models/pixelcnn_v1_hyperparams.json", "r") as f:
    hyperparams = json.load(f)

input_dim = hyperparams["input_dim"]
output_dim = hyperparams["output_dim"]
num_res = hyperparams["residual_blocks"]
filters = hyperparams["filters"]

epochs = hyperparams["epochs"]
batch_size = hyperparams["batch_size"]
learning_rate = 1e-5

### Load Dataset

In [ ]:
image_folder = 'latent_images'

train_loader = LatentIndicesDataset(image_folder, batch_size=batch_size)

Found 200000 latent indices


### Create Model

In [4]:
model = PixelCNN(input_dim, output_dim, num_res, filters).to(device)

### VQ-VAE for Testing

In [5]:
vae = VQVAE(64, 1024, 1)

vae.load_state_dict(torch.load('models/vae_checkpoint_v1.pth', weights_only=True))
vae.to(device);

In [6]:
opt = optimizer(model, learning_rate=learning_rate)

In [7]:
training_state_path = 'state.json'

try:
    with open(training_state_path, 'r') as file:
        training_state_data = json.load(file)
    model.load_state_dict(torch.load(f"pixelcnn_checkpoint_{training_state_data['last_epoch']}.pth", weights_only=True))
except:
    training_state_data = {
        'last_epoch': 0,
        'loss': [],
    }

for epoch in range(training_state_data['last_epoch'] + 1, epochs + 1):
    loss = train(model, train_loader, opt, device)
    
    print(f"\nEpoch [{epoch}/{epochs}]")
    print(f"Loss: {loss:.4f}")
    
    z = model.sample((16, 16), device)
    z = reconstruct_from_indices(z, vae, device)
    save_image(f'output/epoch{epoch}', z)
    
    training_state_data['last_epoch'] = epoch
    training_state_data['loss'].append(loss)
    
    torch.save(model.state_dict(), f"pixelcnn_checkpoint_{epoch}.pth")
    with open(training_state_path, 'w') as state_file:
        json.dump(training_state_data, state_file)

RuntimeError: Expected 3D (unbatched) or 4D (batched) input to conv2d, but got input of size: [16, 16]